In [1]:

!pip install pandas==2.2.2 bitsandbytes accelerate transformers -U

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 15.0 MB/s  0:00:00.3 MB/s eta 0:00:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... error
  error: subprocess-exited-with-error
  
  × Preparing metadata (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [176 lines of output]
      + meson setup /tmp/pip-install-rpvva4lf/pandas_80bf6aa840814cf4a81c0d7ca824c0f0 /tmp/pip-install-rpvva4lf/pandas_80bf6aa840814cf4a81c0d7ca824c0f0/.mesonpy-2ybtg9v_/build -Dbuildtype=release -Db_ndebug=if-release -Db_vscrt=md --vsenv --native-file=/tmp/pip-install-rpvva4lf/pandas_80bf6aa840814cf4a81c0d7ca824c0f0/.mesonpy-2ybtg9v_/build/meson-python-native-file.ini
      The Meson build system
      Version: 1.2.1
      Source dir: /tmp/pip-install-rpvva4lf/pandas_80bf6aa840814cf4a81c0d7ca824c0f0
      Build dir: /tmp/pip-install-rpvva4lf/pandas_80bf6aa840814cf4a8

In [2]:
import pandas as pd
import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

PATH = "./cleaneddata"
df_jd = pd.read_csv(f"{PATH}/jobdesc.csv")
df_res = pd.read_csv(f"{PATH}/resume.csv")
df_know = pd.read_csv(f"{PATH}/knowledge.csv")
df_skills = pd.read_csv(f"{PATH}/skills.csv")
df_tech = pd.read_csv(f"{PATH}/techskills.csv")

print(f"Loaded: {len(df_jd)} JDs, {len(df_res)} Resumes.")
print(f"doine O*NET Federal Database Online.")

ModuleNotFoundError: No module named 'torch'

In [ ]:
model_id = "mistralai/Mistral-7B-Instruct-v0.2"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

print("Meowwww Mistral...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    quantization_config=quant_config,
)
print("Model load.")

Meowwww Mistral...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Model load.


In [ ]:
pip install thefuzz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 46.5 MB/s eta 0:00:00


In [ ]:
from thefuzz import process, fuzz

def get_best_onet_match(job_title, onet_titles):
    match, score = process.extractOne(job_title, onet_titles, scorer=fuzz.token_set_ratio)

    if score > 60:
        return match, score
    return None, 0

In [ ]:
import json
from thefuzz import process, fuzz
from tqdm import tqdm

# 1. The Semantic Bridge: Mapping broad Resume categories to JD keywords
CATEGORY_MAPPING = {
    "INFORMATION-TECHNOLOGY": ["Developer", "Engineer", "Programmer", "System", "DevOps", "Data", "Cloud"],
    "HR": ["Human Resources", "Recruiter", "Administrator", "Manager", "Analyst"],
    "DESIGNER": ["UI", "UX", "Graphics", "Frontend", "Creative", "Designer"],
    "TEACHER": ["Instructor", "Trainer", "Professor", "Education"],
    "ADVOCATE": ["Legal", "Lawyer", "Compliance", "Policy"]
}

tech_lookup = df_tech.groupby('Title')['Example'].apply(list).to_dict()
soft_skill_lookup = df_skills.groupby('Title')['Element Name'].apply(list).to_dict()
onet_titles = list(tech_lookup.keys())

def find_relevant_jd(resume_category, jd_df):
    """Bridges broad categories to specific JDs using fuzzy keyword matching."""
    # Get keywords for the category, default to the category itself if not in map
    keywords = CATEGORY_MAPPING.get(resume_category, [resume_category])

    # Filter JDs where the title contains ANY of the keywords
    pattern = '|'.join(keywords)
    potential_matches = jd_df[jd_df['Job Title'].str.contains(pattern, case=False, na=False)]

    if not potential_matches.empty:
        # Sample one so we don't always match the first 'Flutter' job to every IT resume
        return potential_matches.sample(n=1).iloc[0]
    return None

output_db = []
target_rows = 50 # Bumped this up for a better demo database

# Start from where the data actually gets interesting
for i in tqdm(range(0, len(df_res))):
    if len(output_db) >= target_rows: break

    resume_row = df_res.iloc[i]
    category = resume_row['Category']

    # Use the bridge function to find a JD on the same 'planet'
    jd_row = find_relevant_jd(category, df_jd)
    if jd_row is None: continue

    role_title = jd_row['Job Title']


    onet_match, match_score = process.extractOne(role_title, onet_titles, scorer=fuzz.token_set_ratio)

    if match_score < 55: continue # Lowered threshold slightly for broader tech roles

    prompt = f"""<s>[INST] Extract a flat JSON list of skills from this resume.
    Resume: {resume_row['Resume_str']} [/INST] {{"candidate_skills": ["""

    inputs_res = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs_res = model.generate(
        **inputs_res,
        max_new_tokens=400,
        temperature=0.1,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

    res_response = tokenizer.decode(outputs_res[0], skip_special_tokens=True)
    try:
        raw_json = '{\n  "candidate_skills": [' + res_response.split('{\n  "candidate_skills": [')[-1]
        clean_json = raw_json[:raw_json.rfind('}')+1]
        extracted_skills = json.loads(clean_json)['candidate_skills']
    except Exception:
        continue
    required_skills = set(tech_lookup.get(onet_match, []) + soft_skill_lookup.get(onet_match, []))
    current_skills = set([s.lower().strip() for s in extracted_skills])

    gap = [s for s in required_skills if s.lower() not in current_skills]

    record = {
        "application_id": f"MATCH-{len(output_db)}",
        "resume_id": str(resume_row['ID']),
        "category": category,
        "matched_job": role_title,
        "onet_standard": onet_match,
        "match_score": f"{match_score}%",
        "candidate_skills": extracted_skills,
        "skill_gap": gap[:10], # The core skill gap for the onboarding pathway [cite: 10]
        "reasoning_trace": (
            f"Mapped broad category '{category}' to role '{role_title}'. "
            f"Curriculum grounded in O*NET standard for '{onet_match}'."
        )
    }
    output_db.append(record)

with open("./AI/processedD.json", "w") as f:
    json.dump(output_db, f, indent=4)

  1%|          | 19/2482 [10:52<25:56:45, 37.92s/it]